In [11]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string
from tqdm.autonotebook import tqdm
import numpy as np

In [2]:
data = pd.read_csv('train_data.csv')

In [5]:
model_name = 'multi-qa-MiniLM-L6-cos-v1'
bi_encoder = SentenceTransformer(model_name)
max_seq_len = 512
top_k = 5

Downloading:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/466k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/383 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/349 [00:00<?, ?B/s]

In [6]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Downloading:   0%|          | 0.00/794 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/86.7M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/316 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
passages = data['Paragraph'].unique()
corpus_embeddings = bi_encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True,device='cuda')

Batches:   0%|          | 0/487 [00:00<?, ?it/s]

In [12]:
def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(passages):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

  0%|          | 0/15555 [00:00<?, ?it/s]

In [16]:
index_dict = {}
for i,para in enumerate(data['Paragraph'].unique()):
    index_dict[para] = i
data['para_index'] = data['Paragraph'].map(index_dict)


In [41]:
bm25_results = pd.DataFrame()
questions = data['Question']
original_paras = data['Paragraph']
answer_avail = data['Answer_possible'].apply(lambda x: int(x))

predicted_paras = []
scores_actual_para = []
incorrect_clf = []
top = []
for i in range(len(data)):
    query = data['Question'][i]
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -5)[-5:]
    bm25_hits = [(idx,bm25_scores[idx]) for idx in top_n]
    bm25_hits = sorted(bm25_hits,key=lambda x: -x[1])
    if bm25_hits[0][0] == data['para_index'][i]:
        incorrect_clf.append(0)
        predicted_paras.append(data['Paragraph'][i])
        scores_actual_para.append(bm25_hits[0][1])
        top.append(1)
    elif data['para_index'][i] not in top_n:
        incorrect_clf.append(1)
        predicted_paras.append('')
        scores_actual_para.append(0)
        top.append(-1)
        
    else:
        incorrect_clf.append(1)
        predicted_paras.append(data[data['para_index']==bm25_hits[0][0]].iloc[0,2])
        top.append(bm25_hits.index((data['para_index'][i],bm25_scores[data['para_index'][i]]))+1)
        scores_actual_para.append(bm25_scores[data['para_index'][i]])

bm25_results['questions'] = questions
bm25_results['original_paras'] = original_paras
bm25_results['answer_avail'] = answer_avail
bm25_results['predicted_paras'] = predicted_paras
bm25_results['scores_actual_para'] = scores_actual_para
bm25_results['incorrect_clf'] = incorrect_clf
bm25_results['top'] = top

In [42]:
bm25_results.incorrect_clf.value_counts()

0    48455
1    26600
Name: incorrect_clf, dtype: int64

In [64]:
bm25_results.to_csv('bm25_results.csv')

In [62]:
rerank_results = pd.DataFrame()
questions = data['Question']
original_paras = data['Paragraph']
answer_avail = data['Answer_possible'].apply(lambda x: int(x))

predicted_paras = []
scores_actual_para = []
scores_top_pred = []
incorrect_clf = []
top = []
for i in tqdm(range(len(data))):
    query = data['Question'][i]
    question_embedding = bi_encoder.encode(query, convert_to_tensor=True)
    question_embedding = question_embedding.cuda()
    hits = util.semantic_search(question_embedding, corpus_embeddings, top_k=top_k)
    hits = hits[0]  # Get the hits for the first query

    ##### Re-Ranking #####
    # Now, score all retrieved passages with the cross_encoder
    cross_inp = [[query, passages[hit['corpus_id']]] for hit in hits]
    cross_scores = cross_encoder.predict(cross_inp)
    for idx in range(len(cross_scores)):
        hits[idx]['cross-score'] = cross_scores[idx]

    hits = sorted(hits, key=lambda x: x['cross-score'], reverse=True)
    top_results = [(i['corpus_id'],i['cross-score']) for i in hits]
    scores_top_pred.append(top_results[0][1])
    top_n = [i[0] for i in top_results]
    top_n_scores = [i[1] for i in top_results]
    if top_results[0][0] == data['para_index'][i]:
        incorrect_clf.append(0)
        predicted_paras.append(data['Paragraph'][i])
        scores_actual_para.append(top_results[0][1])
        top.append(1)
    elif data['para_index'][i] not in top_n:
        incorrect_clf.append(1)
        predicted_paras.append('')
        scores_actual_para.append(0)
        top.append(-1)
        
    else:
        incorrect_clf.append(1)
        predicted_paras.append(data[data['para_index']==top_results[0][0]].iloc[0,2])
        top.append(top_n.index(data['para_index'][i])+1)
        scores_actual_para.append(top_n_scores[top[-1]-1])

rerank_results['questions'] = questions
rerank_results['original_paras'] = original_paras
rerank_results['answer_avail'] = answer_avail
rerank_results['predicted_paras'] = predicted_paras
rerank_results['scores_actual_para'] = scores_actual_para
rerank_results['incorrect_clf'] = incorrect_clf
rerank_results['top'] = top

  0%|          | 0/75055 [00:00<?, ?it/s]

In [65]:
rerank_results['incorrect_clf'].value_counts()

0    51278
1    23777
Name: incorrect_clf, dtype: int64

In [63]:
rerank_results.to_csv(f"rerank_results-{model_name}.csv")